In [20]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from dotenv import load_dotenv

In [10]:
load_dotenv()
model = ChatGroq(model="openai/gpt-oss-120b")

In [11]:
model.invoke([HumanMessage(content="Hi my name is Hemanta Ghosh")])

AIMessage(content='Hello Hemanta! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi my name is Hemanta Ghosh". Likely they are introducing themselves. The appropriate response is a friendly greeting, perhaps asking how can I help. Should keep it conversational.'}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 80, 'total_tokens': 147, 'completion_time': 0.138818314, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.003442618, 'prompt_tokens_details': None, 'queue_time': 0.045205768, 'total_time': 0.142260932}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--f3afa0eb-3bfa-4a0c-83b3-b13003d174ea-0', usage_metadata={'input_tokens': 80, 'output_tokens': 67, 'total_tokens': 147, 'output_token_details': {'reasoning': 42}})

In [13]:
result = model.invoke(
    [
        HumanMessage(content="Hi my name is Hemanta Ghosh"),
        AIMessage(content="Hello Hemanta! Nice to meet you. How can I assist you today?"),
        HumanMessage(content="Can you help me with my college assisment ?")
    ]
)

In [14]:
result.content

'Absolutely—I’d be happy to help! Could you let me know a bit more about the assignment?\n\n- **Subject / Course:** What class is it for?  \n- **Type of work:** Is it an essay, problem set, lab report, presentation, coding project, etc.?  \n- **Topic / Prompt:** What specifically are you being asked to address?  \n- **Length / Format requirements:** Any word‑count, page limits, citation style, or other formatting rules?  \n- **Deadline:** When do you need it by?  \n\nThe more details you share, the better I can tailor my assistance to your needs (e.g., outlining, brainstorming ideas, explaining concepts, reviewing drafts, etc.). Feel free to paste the assignment prompt or any notes you have, and we’ll get started!'

In [ ]:
store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]
with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [16]:
config = {"configurable":{"session_id":"chat_1"}}

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi my name is Hemanta Ghosh")],
    config = config
)
response.content

'Hello Hemanta! Nice to meet you. How can I assist you today?'

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config = config
)
response.content

'Your name is Hemanta\u202fGhosh.'

In [19]:
config1 = {"configurable":{"session_id":"chat_2"}}

response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config = config1
)
response.content

'I don’t have any information about your name. If you’d like, feel free to tell me what you’d like to be called!'

### ChatPromptTemplate + Messageholder

In [23]:
prompt = ChatPromptTemplate([


    ("system","You are helpful assitant . Answer all questions to the best of your ability in {language}."),
    MessagesPlaceholder(variable_name="messages"),
]
)
chain = prompt|model

In [24]:
response = chain.invoke({"messages":[HumanMessage(content="Hi how are you what is AI")],"language":"English"})
response.content

'Hello! I’m doing well, thank you for asking. How are you?\n\n**What is AI?**\n\nArtificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include things like:\n\n- **Learning** – recognizing patterns from data (e.g., predicting future trends, understanding speech)\n- **Reasoning** – solving problems, making decisions, or planning actions\n- **Perception** – interpreting sensory information such as images, sound, or text\n- **Language Understanding** – reading, generating, and translating human language\n- **Interaction** – responding to user inputs in a natural, conversational way\n\nAI can be built using various techniques, the most common being **machine learning**, where algorithms improve their performance by being exposed to large amounts of data. A subset of machine learning called **deep learning** uses neural networks with many layers to handle especially compl

In [29]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key = "messages"
)

In [32]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Hemanta")],"language":"Bengali"},
    config=config
)
repsonse.content

'হ্যালো হেমন্ত! আপনার সঙ্গে কথা বলে খুবই আনন্দিত হলাম। আজ আমি কীভাবে আপনার সাহায্য করতে পারি?'

In [33]:
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="What is my name ?")],"language":"Bengali"},
    config=config
)
repsonse.content

'আপনার নাম হেমন্ত।'

## Concept of trimmer 

In [34]:
from langchain_core.messages import SystemMessage,trim_messages

In [35]:
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\HEMANTA GHOSH\Desktop\Langchain\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HEMANTA GHOSH\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [36]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure what your favorite ice‑cream flavor is—could you let me know? Then I can chat more about it!'

In [37]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked for the sum of\u202f2\u202f+\u202f2.'

In [ ]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [39]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You haven’t actually asked a math problem yet—this is the first question in our conversation. If you have a specific math problem you’d like help with, feel free to share it and I’ll be glad to work through it with you!'